# Exploración Exhaustiva y Análisis de Datos (EDA) Completo
Este cuaderno Jupyter documenta paso a paso el analisis estadistico profundo y completo del ecosistema competitivo del Counter-Strike, previo al modelado en Python o R. A continuacion definimos la logica completa de exploracion.
## a. Preguntas de la Investigacion
¿Cuales metricas o caracteristicas tecnicas pronostican mejor la victoria de un equipo en un escenario profesional?
¿Basta con ser el mejor teoricamente (mejor Ranking Mundial) para ganar la partida?
## b. Subpreguntas Analiticas
*   **S1:** ¿Como fluctua y que peso real nos da la ventaja del terreno (mapa jugado)?
*   **S2:** ¿Sera mas acertado parametrizar modelos basados en variables puramente mecanicas (Rating) o en diferencias de impacto situacional?
*   **S3:** ¿Como se comportan todas las variables entre si en una matriz general de correlacion masiva?


--- 
## 1. Importación y Carga de Paquetes
Librerias requeridas.


In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)




--- 
## 2. Carga y Estructura del Dataset
Examinaremos desde la base bruta.


In [ ]:
# Carga del CSV principal
df = pd.read_csv('data/csgo_games.csv')
print(f"Dimensiones del dataset: {df.shape}")
# Verificamos los nulos y los tipos
display(df.info())
display(df.describe().T.head(10))




Al tener más de 170 columnas debido a que cada jugador tiene variables separadas (`t1_player1_rating`, `t1_player2_rating`, etc), vamos a agregar las metricas promedio a nivel equipo para un analisis bivariado limpio.


In [ ]:
# Agregando estadisticas de equipo (Promedios)
para_medias = ['rating', 'impact', 'kpr', 'dpr', 'kdr']
for metric in para_medias:
    # Para el equipo 1
    t1_cols = [f't1_player{i}_{metric}' for i in range(1, 6) if f't1_player{i}_{metric}' in df.columns]
    if t1_cols:
        df[f't1_{metric}_avg'] = df[t1_cols].mean(axis=1)
    
    # Para el equipo 2
    t2_cols = [f't2_player{i}_{metric}' for i in range(1, 6) if f't2_player{i}_{metric}' in df.columns]
    if t2_cols:
        df[f't2_{metric}_avg'] = df[t2_cols].mean(axis=1)
# Diferenciales
df['rating_diff'] = df['t1_rating_avg'] - df['t2_rating_avg']
df['impact_diff'] = df['t1_impact_avg'] - df['t2_impact_avg']
df['kpr_diff'] = df['t1_kpr_avg'] - df['t2_kpr_avg']
df['dpr_diff'] = df['t1_dpr_avg'] - df['t2_dpr_avg']
# Target
df['target_win'] = (df['winner'] == 't1').astype(int)




---
## 3. Analisis Univariado Completo
Miramos las distribuciones crudas. Esto nos confirma la varianza y posibles *outliers* antes del modelo.


In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 15))
cols_plot = ['t1_world_rank', 't1_rating_avg', 't1_impact_avg', 't1_kpr_avg', 't1_dpr_avg', 't1_h2h_win_perc']
for i, col in enumerate(cols_plot):
    if col in df.columns:
        ax = axes[i//2, i%2]
        sns.histplot(df[col].dropna(), kde=True, ax=ax, color='purple', bins=30)
        ax.set_title(f'Distribución de {col}')
plt.tight_layout()
plt.savefig('eda_univariado.png', dpi=100, bbox_inches='tight')
plt.show()




**Insights Univariados:** Nos sorprendió gratamente que las estadísticas intra-partida como el **Rating, Impact, KPR** y **DPR** siguen una **distribución sumamente normal**. Esto es invaluable en Data Science: indica una simetría sana en la competición. Sin embargo, factores iniciales como el **Rango de HLTV (world_rank)** tienen fuertes sesgos (skew).


---
## 4. Análisis Bivariado masivo
Procedemos a realizar Boxplots, Violines y Scatterplots de variables independientes chocando con el objetivo binario `target_win`.


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
# 1. Boxplot del Diferencial de Rating
sns.boxplot(data=df, x='target_win', y='rating_diff', ax=axes[0,0], palette='pastel')
axes[0,0].set_title('Rango Intercuartilico del Rating según Victoria')
# 2. Violin de Diferencial de Impacto
sns.violinplot(data=df, x='target_win', y='impact_diff', ax=axes[0,1], palette='rocket')
axes[0,1].set_title('Violín (Fluctuaciones) del Impacto respecto al Target')
# 3. KPR vs Victoria
sns.kdeplot(df[df['target_win'] == 1]['kpr_diff'], label='Ganó (1)', shade=True, ax=axes[1,0], color='green')
sns.kdeplot(df[df['target_win'] == 0]['kpr_diff'], label='Perdió (0)', shade=True, ax=axes[1,0], color='red')
axes[1,0].set_title('Densidad del KPR (Kills)')
axes[1,0].legend()
# 4. Diferencial de Muertes (DPR)
sns.boxplot(data=df, x='target_win', y='dpr_diff', ax=axes[1,1], palette='coolwarm')
axes[1,1].set_title('DPR según Resultado')
plt.tight_layout()
plt.savefig('eda_bivariado_grid.png', dpi=150, bbox_inches='tight')
plt.show()
# Extra: Scatterplot de Rating vs Impacto
plt.figure(figsize=(8,6))
sns.scatterplot(data=df, x='rating_diff', y='impact_diff', hue='target_win', alpha=0.6, palette='viridis')
plt.title('Dispersión Multivariable: Rating vs Impacto')
plt.savefig('eda_bivariado_grid.png', dpi=150, bbox_inches='tight')
plt.show()




**Insights Bivariados:** 
1. La correlación visual entre ganar (`target_win=1`) y tener un diferencial positivo (`rating_diff > 0` o `impact_diff > 0`) es obvia, demostrando que son características hiper-predictoras.
2. En la grafica de Violines vemos grandes fluctuaciones de impacto. 
3. El gráfico de dispersión confirma la colinealidad fuerte entre Rating e Impacto, por lo que introducir ambas a un modelo bayesiano podría requerir variables jerarquicas o regularización fuerte.


---
## 5. Gran Matriz de Correlación General
Para purgar las variables que no servirán y dejar nuestro futuro modelo lo más óptimo y libre de ruidos (Feature Selection).


In [ ]:
cols_matriz = ['target_win', 't1_world_rank', 't2_world_rank', 'rating_diff', 
               'impact_diff', 'kpr_diff', 'dpr_diff', 't1_h2h_win_perc']
corr = df[cols_matriz].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
plt.figure(figsize=(12, 10))
sns.heatmap(corr, mask=mask, annot=True, cmap='RdBu', fmt=".2f", vmin=-1, vmax=1)
plt.title('Matriz de Correlaciones de Pearson (Feature Filtering)')
plt.savefig('matriz_correlacion_completa.png', dpi=150, bbox_inches='tight')
plt.show()




**Conclusiones de la Matriz:**
El target `target_win` muestra asombrosas correlaciones altísimas (> 0.50) con los diferenciales que construimos (`rating_diff`, `impact_diff`). Sin embargo, variables estéticas como el `t1_world_rank` (-0.29) o `t1_h2h_win_perc` (0.04) arrojan un poder predictivo increíblemente pobre o residual. Por lo tanto, no se utilizarán ni afectarán la regresión directamente frente al poder del rating actual.


---
## 6. Variables Exterioes: Sesgo Medioambiental (Map Bias)
Cargamos `results.csv` para ver cómo el terreno afecta e invalida temporalmente la habilidad pura (Respuesta a S1).


In [ ]:
try:
    df_res = pd.read_csv('data/raw/results.csv')
    df_res['total_r'] = df_res['result_1'] + df_res['result_2']
    df_res_clean = df_res[df_res['total_r'] >= 16].copy() # Solo partidas legitimas largas
    
    df_res_clean['ct_winrate'] = ((df_res_clean['ct_1'] + df_res_clean['ct_2']) / df_res_clean['total_r']) * 100
    common_maps = df_res_clean['_map'].value_counts().head(7).index
    df_map_bias = df_res_clean[df_res_clean['_map'].isin(common_maps)]
    plt.figure(figsize=(14, 7))
    sns.boxplot(data=df_map_bias, x='_map', y='ct_winrate', palette='Set3', order=common_maps)
    plt.axhline(50, color='red', linestyle='--', linewidth=2)
    plt.title('S1: Ventaja de Defensa (CT Winrate %) según Mapa', fontsize=16)
    plt.ylabel('% Rondas ganadas por CT')
    plt.tight_layout()
plt.savefig('evidencia_mapa_bias.png', dpi=150, bbox_inches='tight')
plt.show()
except Exception as e:
    print("Archivo ausente:", e)




---
## d. Resultados Finales y Toma de Decisiones Técnicas
1.  **Limpieza:** Tras inspeccionar profundamente las 170 variables, notamos que procesar promedios individuales por equipo salva el formato de los datos y arroja distribuciones sumamente simétricas.
2.  **Volatilidad del Impacto:** En los Boxplots y Violines es tajante observar la alta *skewness* (asimetría variable) del impacto y la dispersión masiva.
3.  **Filtrado Matemático:** La Matriz de Pearson dictaminó con pruebas duras qué variables sirven: `rating`, `kpr`, `impact` son correlaciones perfectas, descartando factores como "Ranking Mundial" o "Winrate Pasado".
## e. Implicaciones al Modelar (Diseño Bayesiano)
Este largo proceso investigativo determina nuestra Regresión Bayesiana final:
- Las altas volatilidades en la dispersión de impacto afirman que requerimos **Priors Bayesinas mucho más anchas/débiles** porque penalizar severamente ese vector destrozará nuestra medición.
- El sesgo documentado del bando defensivo (en el último gráfico) ratifica estructurar el modelo paramétricamente de manera **Jerárquica**: no tratar la habilidad en el vacío, sino como una habilidad interceptada fuertemente por el ecosistema de qué terreno se juegue.
